
# Junk-vs-Rest CNN Experiment (IF Events)

This notebook runs a full pipeline to classify **junk (1)** vs **not-junk (0)** on IF microscopy *event* images (4-channel crops, typically 75×75), including:

- Loading annotated HDF5 data (`rare_cells_annotated`, `wbcs_annotated`, `junk_annotated`) and unannotated clusters
- **KMeans clustering** of junk → junk strata via **elbow** auto-selection
- **Stratified 60/40** train/val split across rare subtypes, WBC, and junk clusters
- **Non-junk: junk** downsampling in training (default 1.5 : 1; keeps all junk)
- Train a **4-channel CNN** with **Focal Loss**
- **TensorBoard** logging
- **Grad-CAM** (composite & DAPI-only) + **automatic center vs border attention quantification**
- Generate **README.md** and **metadata.json** summarizing the run


In [ ]:

# =========================
# 1) Configuration
# =========================

import os, sys, json, glob, math, time, random, subprocess
from datetime import datetime
from pathlib import Path

ROOT = "/mnt/deepstore/Vidur/Junk Classification/data_model_labels"  # <-- change if needed
EPOCHS = 10
BATCH_SIZE = 256
LR = 1e-3
FOCAL_ALPHA = 0.25
FOCAL_GAMMA = 2.0
USE_AMP = True

NONJUNK_TO_JUNK = 1.5   # keep all junk; downsample non-junk to this ratio
VAL_FRAC = 0.40         # 60/40 split
JUNK_KMAX = 20           # evaluate k in [2..KMAX] for elbow
JUNK_KMEANS_CAP = None  # optional cap on per-file rows used to extract junk features for KMeans (None = all)

TARGET_HW = 75
SEED = 42

OUT = f"experiment_outputs_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(OUT, exist_ok=True)

print("Output directory:", OUT)

Output directory: experiment_outputs_20251024_150445


In [ ]:
# =========================
# 2) Imports & AMP compat
# =========================

import h5py
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

from sklearn.cluster import KMeans
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

# AMP compatibility (works on old/new PyTorch; no warnings)
try:
    from torch import amp
    autocast = amp.autocast
    GradScaler = amp.GradScaler
except ImportError:
    from torch.cuda.amp import autocast, GradScaler

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [ ]:
# =========================
# 3) Helpers (files, elbow, features)
# =========================

def find_h5s(root, subfolder):
    d = os.path.join(root, subfolder)
    return sorted(glob.glob(os.path.join(d, "*.hdf5")))

def choose_k_by_elbow(inertias, ks):
    # Distance from line between first and last
    x1, y1 = ks[0], inertias[0]
    x2, y2 = ks[-1], inertias[-1]
    best_k = ks[0]; best_dist = -1
    for k, y in zip(ks, inertias):
        num = abs((y2 - y1)*k - (x2 - x1)*y + x2*y1 - y2*x1)
        den = math.sqrt((y2 - y1)**2 + (x2 - x1)**2) + 1e-9
        d = num / den
        if d > best_dist:
            best_dist = d; best_k = k
    return best_k

def junk_features_h5(path, max_rows=None, target_hw=75):
    feats = []
    idxs  = []
    with h5py.File(path, "r") as f:
        X = f["images"]
        N = X.shape[0]
        take = N if max_rows is None else min(max_rows, N)
        for i in range(take):
            img = X[i].astype(np.float32)  # HWC
            H,W,C = img.shape
            if (H,W)!=(target_hw,target_hw):
                t = torch.from_numpy(img).permute(2,0,1).unsqueeze(0).float()
                t = F.interpolate(t, size=(target_hw,target_hw), mode="bilinear", align_corners=False)
                img = t.squeeze(0).permute(1,2,0).numpy()
            # per-channel mean/std + center-border contrast
            ch_mean = img.reshape(-1, C).mean(axis=0)
            ch_std  = img.reshape(-1, C).std(axis=0)
            r = 10
            cx, cy = target_hw//2, target_hw//2
            center = img[cx-r:cx+r+1, cy-r:cy+r+1, :].reshape(-1, C).mean(axis=0)
            mask = np.ones((target_hw, target_hw), dtype=bool)
            mask[cx-r:cx+r+1, cy-r:cy+r+1] = False
            border = img[mask].reshape(-1, C).mean(axis=0)
            contrast = center - border
            feats.append(np.concatenate([ch_mean, ch_std, contrast], 0))
            idxs.append(i)
    return np.array(feats, np.float32), np.array(idxs, np.int32)


In [ ]:

# =========================
# 4) Build annotated index & KMeans on junk
# =========================

def make_rgb(arr):
        # Convert 4-channel array -> RGB for visualization (customize as needed)
        # Example: [DAPI, FITC, TRITC, CY5] → Blue, Green, Red, Magenta
        arr = arr / (arr.max() + 1e-8)
        rgb = np.zeros((arr.shape[0], arr.shape[1], 3), dtype=np.float32)
        if arr.shape[-1] >= 3:
            rgb[..., 0] = np.clip(arr[..., 2] + 0.3*arr[..., 3], 0, 1)  # Red + Cy5
            rgb[..., 1] = np.clip(arr[..., 1], 0, 1)                    # FITC
            rgb[..., 2] = np.clip(arr[..., 0], 0, 1)                    # DAPI
        else:
            rgb[..., :] = arr[..., [0,0,0]]  # fallback grayscale
        return rgb

def build_annotated_index(root, target_hw=75, junk_k_max=8, junk_kmeans_cap=None, seed=42):
    rare_files = find_h5s(root, "rare_cells_annotated")
    wbc_files  = find_h5s(root, "wbcs_annotated")
    junk_files = find_h5s(root, "junk_annotated")

    if not junk_files: raise RuntimeError("No junk files found.")
    if not (rare_files or wbc_files): raise RuntimeError("Need rare and/or WBC files.")

    items = []
    # rare subtypes as separate strata
    for p in rare_files:
        stem = os.path.splitext(os.path.basename(p))[0]
        with h5py.File(p,"r") as f: N = f["images"].shape[0]
        for r in range(N):
            items.append((p, r, 0, f"rare:{stem}"))
    # wbc
    for p in wbc_files:
        with h5py.File(p,"r") as f: N = f["images"].shape[0]
        for r in range(N):
            items.append((p, r, 0, "wbc"))
    # junk (temp meta)
    for p in junk_files:
        with h5py.File(p,"r") as f: N = f["images"].shape[0]
        for r in range(N):
            items.append((p, r, 1, "junk:unknown"))

    # KMeans on junk
    feat_list=[]; row_refs=[]
    for p in junk_files:
        feats, idxs = junk_features_h5(p, max_rows=junk_kmeans_cap, target_hw=target_hw)
        for j,local_idx in enumerate(idxs):
            feat_list.append(feats[j]); row_refs.append((p, int(local_idx)))
    X = np.vstack(feat_list).astype(np.float32)
    ks = list(range(2, max(3, junk_k_max+1)))
    inertias=[]
    for k in ks:
        km = KMeans(n_clusters=k, n_init=10, random_state=seed)
        km.fit(X); inertias.append(km.inertia_)
    k_star = choose_k_by_elbow(inertias, ks)

    # Save elbow plot
    plt.figure(figsize=(5,4))
    plt.plot(ks, inertias, marker="o")
    plt.xlabel("k"); plt.ylabel("Inertia"); plt.title("Junk KMeans Elbow")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, "junk_elbow_plot.png"), dpi=150)
    plt.close()

    km = KMeans(n_clusters=k_star, n_init=20, random_state=seed)
    labels_k = km.fit_predict(X)
    junk_cluster_map = {row_refs[i]: int(labels_k[i]) for i in range(len(row_refs))}

    # update meta
    items2 = []
    for (p,r,lab,meta) in items:
        if lab==1:
            c = junk_cluster_map.get((p,r), -1)
            items2.append((p,r,lab, f"junk:c{c}" if c>=0 else "junk:cNA"))
        else:
            items2.append((p,r,lab,meta))

    # thumbnails per cluster (most & least representative)
    # compute distances to cluster centroids
    distances = km.transform(X)  # (n_samples, k_star)
    for c in range(k_star):
        # Sort samples by distance to centroid
        dist_c = distances[:, c]
        idx_c = np.where(labels_k == c)[0]
        sort_idx = np.argsort(dist_c[idx_c])
        top_idx = idx_c[sort_idx[:100]]
        bot_idx = idx_c[sort_idx[-100:]]

        for subset, tag in [(top_idx, "most"), (bot_idx, "least")]:
            examples = [row_refs[i] for i in subset]
            imgs = []
            for (pp, rr) in examples:
                with h5py.File(pp, "r") as f:
                    arr = f["images"][rr].astype(np.float32)
                H, W, C = arr.shape
                if (H, W) != (target_hw, target_hw):
                    t = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).float()
                    t = F.interpolate(t, size=(target_hw, target_hw), mode="bilinear", align_corners=False)
                    arr = t.squeeze(0).permute(1, 2, 0).numpy()
                imgs.append(make_rgb(arr))

            n = len(imgs)
            cols = 10
            rows = (n + cols - 1) // cols
            plt.figure(figsize=(cols * 1.2, rows * 1.2))
            for i, img in enumerate(imgs):
                plt.subplot(rows, cols, i + 1)
                plt.imshow(img)
                plt.axis("off")
            plt.suptitle(f"Junk Cluster c{c} ({tag} representative)")
            plt.tight_layout()
            plt.savefig(os.path.join(OUT, f"junk_cluster_c{c}_{tag}_rep.png"), dpi=150)
            plt.close()


    return items2, k_star


In [ ]:

# =========================
# 5) Split 60/40 & balance
# =========================

def stratified_split_and_balance(items, val_frac=0.40, nonjunk_to_junk=1.5, seed=42):
    rng = np.random.default_rng(seed)
    metas = np.array([it[3] for it in items])
    idx = np.arange(len(items))

    splitter = StratifiedShuffleSplit(n_splits=1, test_size=val_frac, random_state=seed)
    tr_idx, va_idx = next(splitter.split(idx, metas))
    train_items = [items[i] for i in tr_idx]
    val_items   = [items[i] for i in va_idx]

    # Downsample non-junk in train to target ratio (keep all junk)
    train_y = np.array([it[2] for it in train_items], int)
    nj_idx = [i for i,y in enumerate(train_y) if y==0]
    j_idx  = [i for i,y in enumerate(train_y) if y==1]
    n_junk = len(j_idx)
    target_nonjunk = int(nonjunk_to_junk * n_junk)

    if len(nj_idx) > target_nonjunk:
        # per-stratum proportional sampling
        nj_items = [train_items[i] for i in nj_idx]
        nj_meta  = np.array([it[3] for it in nj_items])
        counts = {m: (nj_meta==m).sum() for m in np.unique(nj_meta)}
        selected = []
        for m, cnt in counts.items():
            share = max(1, int(round(target_nonjunk * (cnt / len(nj_items)))))
            cand = np.where(nj_meta==m)[0]
            pick = rng.choice(cand, size=min(share, cnt), replace=False)
            selected.extend(pick.tolist())
        # trim to exact target size
        if len(selected) > target_nonjunk:
            selected = rng.choice(np.array(selected), size=target_nonjunk, replace=False).tolist()
        balanced_nonjunk = [nj_items[i] for i in selected]
        balanced_train = balanced_nonjunk + [train_items[i] for i in j_idx]
    else:
        balanced_train = train_items

    rng.shuffle(balanced_train)
    return balanced_train, val_items


In [ ]:

# =========================
# 6) Dataset & channel stats
# =========================

class H5BinaryDataset(Dataset):
    def __init__(self, items, ch_mean, ch_std, target_hw=75, dtype=torch.float32):
        self.items = items
        self.ch_mean = ch_mean; self.ch_std = ch_std
        self.target_hw = target_hw; self.dtype = dtype
        self._handles = {}

    def __len__(self): return len(self.items)

    def _open(self, path):
        if path in self._handles: return self._handles[path]
        f = h5py.File(path, "r"); d = f["images"]
        chans=None
        if "channels" in f:
            raw=f["channels"][()]
            try:
                chans=[c.decode() if isinstance(c,(bytes,bytearray)) else str(c) for c in raw]
            except: pass
        self._handles[path]=(f,d,chans); return self._handles[path]

    def __getitem__(self, i):
        path, ridx, lab, _ = self.items[i]
        f, d, _ = self._open(path)
        img = d[ridx].astype(np.float32)  # HWC
        H,W,C = img.shape
        if (H,W)!=(self.target_hw,self.target_hw):
            t = torch.from_numpy(img).permute(2,0,1).unsqueeze(0).float()
            t = F.interpolate(t, size=(self.target_hw,self.target_hw), mode="bilinear", align_corners=False)
            img = t.squeeze(0).permute(1,2,0).numpy()
        for c in range(C):
            img[...,c] = (img[...,c]-self.ch_mean[c])/(self.ch_std[c]+1e-8)
        x = torch.from_numpy(img).permute(2,0,1).to(self.dtype)
        y = torch.tensor(int(lab), dtype=torch.long)
        return x, y

    def close(self):
        for p,(f,_,_) in self._handles.items():
            try: f.close()
            except: pass
        self._handles.clear()

def estimate_channel_stats(items, max_samples=6000, target_hw=75):
    rng = np.random.default_rng(0)
    if len(items)==0: return [0,0,0,0],[1,1,1,1]
    take = min(max_samples, len(items))
    picks = rng.choice(np.arange(len(items)), size=take, replace=False)
    sums=None; sums2=None; cnt=0; C=None
    for i in picks:
        path, ridx, _, _ = items[i]
        with h5py.File(path,"r") as f:
            arr = f["images"][ridx].astype(np.float32)
        H,W,Ctmp = arr.shape; C=C or Ctmp
        if (H,W)!=(target_hw,target_hw):
            t = torch.from_numpy(arr).permute(2,0,1).unsqueeze(0).float()
            t = F.interpolate(t, size=(target_hw,target_hw), mode="bilinear", align_corners=False)
            arr = t.squeeze(0).permute(1,2,0).numpy()
        ch_means = arr.reshape(-1, C).mean(axis=0)
        ch_vars  = arr.reshape(-1, C).var(axis=0)
        if sums is None:
            sums=np.zeros((C,),np.float64); sums2=np.zeros((C,),np.float64)
        sums+=ch_means; sums2+=ch_vars+ch_means**2; cnt+=1
    mean = (sums/cnt).astype(np.float32)
    ex2  = (sums2/cnt).astype(np.float32)
    std  = np.sqrt(np.maximum(ex2 - mean**2, 1e-8)).astype(np.float32)
    return mean.tolist(), std.tolist()


In [ ]:

# =========================
# 7) Model, Focal Loss, Train/Eval
# =========================

class SimpleCNN(nn.Module):
    def __init__(self, in_ch=4):
        super().__init__()
        self.feat = nn.Sequential(
            nn.Conv2d(in_ch, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64,128,3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.fc   = nn.Linear(128, 2)

    def forward(self, x):
        x = self.feat(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)

class BinaryFocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha=alpha; self.gamma=gamma; self.reduction=reduction
    def forward(self, logits, target):
        ce = F.cross_entropy(logits, target, reduction='none')
        pt = torch.softmax(logits, dim=1).gather(1, target.view(-1,1)).squeeze(1)
        alpha_t = torch.where(target==1, self.alpha, 1.0-self.alpha)
        loss = alpha_t * (1-pt).pow(self.gamma) * ce
        if self.reduction=="mean": return loss.mean()
        if self.reduction=="sum":  return loss.sum()
        return loss

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    ys=[]; ps=[]
    for xb,yb in loader:
        xb = xb.to(device); yb = yb.to(device)
        prob1 = torch.softmax(model(xb), dim=1)[:,1]
        ys.append(yb.cpu().numpy()); ps.append(prob1.cpu().numpy())
    y = np.concatenate(ys); p = np.concatenate(ps)
    pred = (p>=0.5).astype(int)
    acc = accuracy_score(y, pred)
    try: auc = roc_auc_score(y, p)
    except: auc = float('nan')
    f1 = f1_score(y, pred, zero_division=0)
    return acc, auc, f1

def train_model(train_items, val_items, ch_mean, ch_std, *, out_dir, epochs, batch_size, lr, alpha, gamma, use_amp, device):
    train_ds = H5BinaryDataset(train_items, ch_mean, ch_std, target_hw=TARGET_HW)
    val_ds   = H5BinaryDataset(val_items,   ch_mean, ch_std, target_hw=TARGET_HW)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    model = SimpleCNN(in_ch=4).to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=lr)
    criterion = BinaryFocalLoss(alpha=alpha, gamma=gamma)
    writer = SummaryWriter(log_dir=os.path.join(out_dir, "tensorboard"))

    scaler = GradScaler(enabled=use_amp)
    best = {"acc":-1,"auc":float('nan'),"f1":-1,"state":None}
    hist = []

    t0=time.time()
    for ep in range(1, epochs+1):
        model.train(); tot=0; n=0
        for xb,yb in train_loader:
            xb=xb.to(device); yb=yb.to(device)
            optim.zero_grad(set_to_none=True)
            with autocast(device_type="cuda", enabled=use_amp):
                logits = model(xb)
                loss = criterion(logits, yb)
            scaler.scale(loss).backward()
            scaler.step(optim); scaler.update()
            tot += loss.item()*xb.size(0); n += xb.size(0)

        acc, auc, f1 = evaluate(model, val_loader, device)
        writer.add_scalar("loss/train", tot/max(n,1), ep)
        writer.add_scalar("metrics/val_acc", acc, ep)
        writer.add_scalar("metrics/val_auc", auc, ep)
        writer.add_scalar("metrics/val_f1",  f1,  ep)
        hist.append({"epoch":ep,"train_loss":tot/max(n,1),"val_acc":acc,"val_auc":auc,"val_f1":f1})
        print(f"[ep {ep}/{epochs}] loss={tot/max(n,1):.4f} | val_acc={acc:.3f} auc={auc:.3f} f1={f1:.3f}")
        if acc > best["acc"]:
            best = {"acc":acc,"auc":auc,"f1":f1,"state":{k:v.cpu() for k,v in model.state_dict().items()}}
    writer.close()
    print("Training time:", round(time.time()-t0,1),"s")
    model.load_state_dict(best["state"])
    return model, best, hist, (train_ds, val_ds)


In [ ]:

# =========================
# 8) Grad-CAM + Center/Border quant
# =========================

class GradCAM:
    def __init__(self, model):
        self.model = model.eval()
        self.target_module = self.model.feat[-2]  # last Conv2d(64->128)
        self._acts=None; self._grads=None
        self.h1 = self.target_module.register_forward_hook(self._fw)
        self.h2 = self.target_module.register_full_backward_hook(self._bw)
    def _fw(self, m, i, o): self._acts = o.detach()
    def _bw(self, m, gin, gout): self._grads = gout[0].detach()
    def _heatmap(self):
        w = self._grads.mean(dim=(2,3), keepdim=True)
        cam = (w * self._acts).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        vmin = cam.view(cam.size(0), -1).min(dim=1)[0].view(-1,1,1,1)
        vmax = cam.view(cam.size(0), -1).max(dim=1)[0].view(-1,1,1,1) + 1e-6
        cam = (cam - vmin) / (vmax - vmin)
        return cam
    def generate(self, xb, target_class=1):
        xb = xb.requires_grad_(True)
        logits = self.model(xb)
        score = logits[:,target_class].sum()
        self.model.zero_grad(set_to_none=True)
        score.backward(retain_graph=True)
        return self._heatmap()
    def close(self):
        try: self.h1.remove(); self.h2.remove()
        except: pass

def find_dapi_index(dataset, fallback=0):
    # Inspect channels from any file in dataset
    for path, ridx, lab, meta in dataset.items[:500]:
        with h5py.File(path,"r") as f:
            if "channels" in f:
                raw=f["channels"][()]
                chans=[c.decode() if isinstance(c,(bytes,bytearray)) else str(c) for c in raw]
                for i,c in enumerate(chans):
                    if str(c).upper().startswith("DAPI"):
                        return i
    return fallback

def load_batch_for_cam(dataset, indices, ch_mean, ch_std, dapi_only=False, dapi_idx=0, target_hw=75, device="cuda"):
    xs=[]; ys=[]
    for i in indices:
        path,ridx,lab,meta = dataset.items[i]
        with h5py.File(path,"r") as f:
            img=f["images"][ridx].astype(np.float32)
        H,W,C = img.shape
        if (H,W)!=(target_hw,target_hw):
            t=torch.from_numpy(img).permute(2,0,1).unsqueeze(0).float()
            t=F.interpolate(t,size=(target_hw,target_hw),mode="bilinear",align_corners=False)
            img=t.squeeze(0).permute(1,2,0).numpy()
        for c in range(C):
            img[...,c]=(img[...,c]-ch_mean[c])/(ch_std[c]+1e-8)
        if dapi_only:
            tmp=np.zeros_like(img); tmp[...,dapi_idx]=img[...,dapi_idx]; img=tmp
        xs.append(torch.from_numpy(img).permute(2,0,1)); ys.append(lab)
    xb=torch.stack(xs,0).to(device)
    yb=torch.tensor(ys).to(device)
    return xb,yb

def center_border_ratio(cam, center_radius=12):
    # cam: (N,1,H,W) in [0,1]
    N,_,H,W = cam.shape
    cx, cy = H//2, W//2
    yy, xx = torch.meshgrid(torch.arange(H, device=cam.device), torch.arange(W, device=cam.device), indexing="ij")
    dist2 = (yy-cx)**2 + (xx-cy)**2
    center_mask = (dist2 <= center_radius**2).float()[None,None,:,:]
    border_mask = 1.0 - center_mask
    center = (cam*center_mask).sum(dim=(2,3)) / (center_mask.sum(dim=(2,3))+1e-6)  # (N,1)
    border = (cam*border_mask).sum(dim=(2,3)) / (border_mask.sum(dim=(2,3))+1e-6)
    ratio = (center / (border+1e-6)).squeeze(1)  # (N,)
    return ratio

def gradcam_reports(model, dataset, out_dir, ch_mean, ch_std, device="cuda", samples_per_class=32, pdf_prefix="gradcam_val"):
    os.makedirs(out_dir, exist_ok=True)
    # infer DAPI index
    dapi_idx = find_dapi_index(dataset, fallback=0)
    # sample balanced indices
    y = np.array([lab for (_,_,lab,_) in dataset.items])
    rng = np.random.default_rng(0)
    idx0 = np.where(y==0)[0]; idx1 = np.where(y==1)[0]
    pick0 = rng.choice(idx0, size=min(samples_per_class, len(idx0)), replace=False) if len(idx0)>0 else []
    pick1 = rng.choice(idx1, size=min(samples_per_class, len(idx1)), replace=False) if len(idx1)>0 else []

    cammer = GradCAM(model)

    # (A) Composite CAM
    import matplotlib.backends.backend_pdf as pdf
    pdf_path = os.path.join(out_dir, f"{pdf_prefix}_composite.pdf")
    p = pdf.PdfPages(pdf_path)
    for group, name in [(pick0,"notjunk"), (pick1,"junk")]:
        if len(group)==0: continue
        xb, yb = load_batch_for_cam(dataset, group, ch_mean, ch_std, dapi_only=False, dapi_idx=dapi_idx, target_hw=TARGET_HW, device=device)
        cams = cammer.generate(xb, target_class=1)  # (N,1,H,W)
        xs = xb.detach().cpu().numpy()
        cm = cams.detach().cpu()
        ratios = center_border_ratio(cm, center_radius=12).cpu().numpy()
        for i in range(xs.shape[0]):
            img_arr = xs[i].transpose(1,2,0)
            rgb = make_rgb(img_arr)  # reuse same make_rgb() helper as above
            hm = cm[i,0].numpy()
            plt.figure(figsize=(9,3))
            plt.subplot(1,3,1)
            plt.imshow(rgb); plt.title(f"{name} (RGB)"); plt.axis("off")
            plt.subplot(1,3,2)
            plt.imshow(rgb); plt.imshow(hm, alpha=0.45, cmap="jet"); plt.title(f"GradCAM (ratio={ratios[i]:.2f})"); plt.axis("off")
            plt.subplot(1,3,3)
            plt.imshow(hm, cmap="jet"); plt.title("CAM heatmap"); plt.axis("off")
            p.savefig(); plt.close()
    p.close()

    # (B) DAPI-only CAM
    pdf_path2 = os.path.join(out_dir, f"{pdf_prefix}_dapi_only.pdf")
    p2 = pdf.PdfPages(pdf_path2)
    for group, name in [(pick0,"notjunk"), (pick1,"junk")]:
        if len(group)==0: continue
        xb, yb = load_batch_for_cam(dataset, group, ch_mean, ch_std, dapi_only=True, dapi_idx=dapi_idx, target_hw=TARGET_HW, device=device)
        cams = cammer.generate(xb, target_class=1)
        xs = xb.detach().cpu().numpy()
        cm = cams.detach().cpu()
        ratios = center_border_ratio(cm, center_radius=12).cpu().numpy()
        for i in range(xs.shape[0]):
            img = xs[i, dapi_idx]
            hm = cm[i,0].numpy()
            plt.subplot(1,3,1)
            plt.imshow(np.stack([np.zeros_like(img), np.zeros_like(img), img], axis=-1))
            plt.title(f"{name} (DAPI)"); plt.axis("off")
            p2.savefig(); plt.close()
    p2.close()

    # (C) Quantitative summary on a larger sample
    # Use up to 1024 random validation samples
    all_idx = np.arange(len(dataset.items))
    rng = np.random.default_rng(1)
    sel = rng.choice(all_idx, size=min(1024, len(all_idx)), replace=False) if len(all_idx)>0 else []
    xb, yb = load_batch_for_cam(dataset, sel, ch_mean, ch_std, dapi_only=False, dapi_idx=dapi_idx, target_hw=TARGET_HW, device=device)
    cams = cammer.generate(xb, target_class=1)
    ratios = center_border_ratio(cams, center_radius=12).detach().cpu().numpy()
    y_np = yb.cpu().numpy()
    report = {
        "center_border_ratio_mean": float(np.mean(ratios)) if len(ratios)>0 else None,
        "center_border_ratio_median": float(np.median(ratios)) if len(ratios)>0 else None,
        "per_class": {
            "notjunk": float(np.mean(ratios[y_np==0])) if np.any(y_np==0) else None,
            "junk":    float(np.mean(ratios[y_np==1])) if np.any(y_np==1) else None,
        },
        "samples": int(len(ratios)),
        "dapi_channel_index": int(dapi_idx),
    }
    with open(os.path.join(out_dir, "gradcam_center_border_summary.json"), "w") as f:
        json.dump(report, f, indent=2)

    cammer.close()
    return report


In [ ]:

# =========================
# 9) Predict on unannotated clusters
# =========================

@torch.no_grad()
def predict_unannotated_and_plot(model, root, ch_mean, ch_std, out_dir, device="cuda"):
    files = find_h5s(root, "unannotated")
    if not files:
        print("No unannotated files found.")
        return {}
    stats = {}
    for p in files:
        with h5py.File(p,"r") as f:
            imgs = f["images"]; N = imgs.shape[0]
            bs = 512; preds=[]
            for s in range(0, N, bs):
                e = min(N, s+bs)
                Xb=[]
                for i in range(s,e):
                    arr = imgs[i].astype(np.float32)
                    H,W,C = arr.shape
                    if (H,W)!=(TARGET_HW,TARGET_HW):
                        t = torch.from_numpy(arr).permute(2,0,1).unsqueeze(0).float()
                        t = F.interpolate(t, size=(TARGET_HW,TARGET_HW), mode="bilinear", align_corners=False)
                        arr = t.squeeze(0).permute(1,2,0).numpy()
                    for c in range(C):
                        arr[...,c] = (arr[...,c]-ch_mean[c])/(ch_std[c]+1e-8)
                    Xb.append(torch.from_numpy(arr).permute(2,0,1))
                xb = torch.stack(Xb,0).to(device)
                prob1 = torch.softmax(model(xb), dim=1)[:,1].cpu().numpy()
                preds.extend((prob1>=0.5).astype(np.uint8).tolist())
        from collections import Counter
        c = Counter(preds)
        stats[os.path.basename(p)] = {"n":len(preds), "junk":int(c.get(1,0)), "notjunk":int(c.get(0,0))}
    # plot
    names = sorted(stats.keys())
    junk = [stats[n]["junk"]/max(stats[n]["n"],1) for n in names]
    notj = [stats[n]["notjunk"]/max(stats[n]["n"],1) for n in names]
    x = np.arange(len(names)); width=0.4
    plt.figure(figsize=(10,5))
    plt.bar(x-width/2, notj, width=width, label="not junk (0)")
    plt.bar(x+width/2, junk,  width=width, label="junk (1)")
    plt.xticks(x, names, rotation=45, ha="right")
    plt.ylabel("Proportion"); plt.title("Predicted label proportions per MM_cluster file")
    plt.legend(); plt.tight_layout()
    plot_path = os.path.join(out_dir, "unannotated_label_proportions.png")
    plt.savefig(plot_path, dpi=150); plt.close()
    with open(os.path.join(out_dir,"unannotated_label_counts.json"),"w") as f:
        json.dump(stats, f, indent=2)
    return stats


In [ ]:

# =========================
# 10) Run pipeline
# =========================

# (A) Build annotated index & KMeans on junk
items, k_star = build_annotated_index(
    ROOT, target_hw=TARGET_HW,
    junk_k_max=JUNK_KMAX, junk_kmeans_cap=JUNK_KMEANS_CAP,
    seed=SEED
)
print("Elbow-selected K for junk clusters:", k_star)

# (B) Channel stats
ch_mean, ch_std = estimate_channel_stats(items, max_samples=6000, target_hw=TARGET_HW)
print("Channel mean:", [round(x,2) for x in ch_mean])
print("Channel std :", [round(x,2) for x in ch_std])

# (C) Split & balance
train_items, val_items = stratified_split_and_balance(
    items, val_frac=VAL_FRAC, nonjunk_to_junk=NONJUNK_TO_JUNK, seed=SEED
)
n_tr = len(train_items); n_va = len(val_items)
n_tr_j = sum(1 for it in train_items if it[2]==1)
n_tr_nj= n_tr - n_tr_j
print(f"Train: {n_tr}  (junk={n_tr_j}, notjunk={n_tr_nj}, ratio={n_tr_nj/max(n_tr_j,1):.2f})")
print(f"Val:   {n_va}")

# (D) Train
model, best, hist, datasets = train_model(
    train_items, val_items, ch_mean, ch_std,
    out_dir=OUT, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR,
    alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA, use_amp=USE_AMP, device=device
)
print("Best val metrics:", best)

# save model
torch.save({
    "state_dict": {k:v.cpu() for k,v in model.state_dict().items()},
    "mean": ch_mean, "std": ch_std,
    "best_acc": best["acc"], "best_auc": best["auc"], "best_f1": best["f1"]
}, os.path.join(OUT, "best_binary_cnn.pt"))


Elbow-selected K for junk clusters: 4
Channel mean: [2233.25, 2629.92, 6724.07, 1936.89]
Channel std : [5507.65, 7083.78, 11619.25, 6501.74]
Train: 12904  (junk=5162, notjunk=7742, ratio=1.50)
Val:   18086
[ep 1/10] loss=0.0446 | val_acc=0.898 auc=0.964 f1=0.679
[ep 2/10] loss=0.0307 | val_acc=0.925 auc=0.975 f1=0.775
[ep 3/10] loss=0.0239 | val_acc=0.944 auc=0.982 f1=0.846
[ep 4/10] loss=0.0207 | val_acc=0.949 auc=0.986 f1=0.852
[ep 5/10] loss=0.0190 | val_acc=0.952 auc=0.987 f1=0.876
[ep 6/10] loss=0.0180 | val_acc=0.952 auc=0.987 f1=0.858
[ep 7/10] loss=0.0182 | val_acc=0.944 auc=0.984 f1=0.860
[ep 8/10] loss=0.0165 | val_acc=0.967 auc=0.993 f1=0.913
[ep 9/10] loss=0.0140 | val_acc=0.959 auc=0.992 f1=0.884
[ep 10/10] loss=0.0137 | val_acc=0.974 auc=0.995 f1=0.931
Training time: 215.6 s
Best val metrics: {'acc': 0.9743447970806148, 'auc': 0.9949227103024308, 'f1': 0.9314015375517445, 'state': {'feat.0.weight': tensor([[[[ 0.1211,  0.1379, -0.0360],
          [ 0.1492, -0.0347,  0.037

In [ ]:

# =========================
# 11) Grad-CAM reports + quantification
# =========================

train_ds, val_ds = datasets
report = gradcam_reports(
    model, val_ds, OUT, ch_mean, ch_std, device=device,
    samples_per_class=32, pdf_prefix="gradcam_val"
)
print("Grad-CAM center/border report:", json.dumps(report, indent=2))


Grad-CAM center/border report: {
  "center_border_ratio_mean": 24812.212890625,
  "center_border_ratio_median": 10100.689453125,
  "per_class": {
    "notjunk": 20531.982421875,
    "junk": 42337.73046875
  },
  "samples": 1024,
  "dapi_channel_index": 0
}


In [ ]:

# =========================
# 12) Predict on unannotated clusters
# =========================

cluster_stats = predict_unannotated_and_plot(model, ROOT, ch_mean, ch_std, OUT, device=device)
print("Cluster stats:", json.dumps(cluster_stats, indent=2)[:500], "...")


Cluster stats: {
  "MM_cluster_0.hdf5": {
    "n": 28162,
    "junk": 2000,
    "notjunk": 26162
  },
  "MM_cluster_1.hdf5": {
    "n": 25563,
    "junk": 2608,
    "notjunk": 22955
  },
  "MM_cluster_2.hdf5": {
    "n": 25489,
    "junk": 4866,
    "notjunk": 20623
  },
  "MM_cluster_3.hdf5": {
    "n": 22531,
    "junk": 2981,
    "notjunk": 19550
  },
  "MM_cluster_4.hdf5": {
    "n": 8903,
    "junk": 464,
    "notjunk": 8439
  },
  "MM_cluster_5.hdf5": {
    "n": 6919,
    "junk": 1002,
    "notjunk": 591 ...


In [ ]:

# =========================
# 13) Save curves, README, metadata
# =========================

# training curves
import pandas as pd
hist_df = pd.DataFrame(hist)
hist_df.to_csv(os.path.join(OUT, "metrics_over_epochs.csv"), index=False)

plt.figure(figsize=(6,4))
plt.plot(hist_df["epoch"], hist_df["train_loss"], marker="o", label="train loss")
plt.plot(hist_df["epoch"], hist_df["val_acc"], marker="o", label="val acc")
plt.plot(hist_df["epoch"], hist_df["val_auc"], marker="o", label="val auc")
plt.plot(hist_df["epoch"], hist_df["val_f1"],  marker="o", label="val f1")
plt.xlabel("epoch"); plt.ylabel("value"); plt.title("Learning Curves")
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUT, "train_curve.png"), dpi=150)
plt.close()

# confusion matrix
from sklearn.metrics import confusion_matrix
# build val loader quickly
val_loader = DataLoader(val_ds, batch_size=512, shuffle=False, num_workers=2, pin_memory=True)
ys=[]; ps=[]
with torch.no_grad():
    for xb,yb in val_loader:
        xb=xb.to(device); yb=yb.to(device)
        prob1 = torch.softmax(model(xb), dim=1)[:,1]
        ys.append(yb.cpu().numpy()); ps.append(prob1.cpu().numpy())
y = np.concatenate(ys); p = np.concatenate(ps); yhat=(p>=0.5).astype(int)
cm = confusion_matrix(y, yhat, labels=[0,1])
plt.figure(figsize=(4,4))
plt.imshow(cm)
plt.xticks([0,1], ["notjunk","junk"]); plt.yticks([0,1], ["notjunk","junk"])
plt.title("Confusion Matrix (val)")
for i in range(2):
    for j in range(2):
        plt.text(j, i, str(cm[i,j]), ha="center", va="center")
plt.tight_layout()
plt.savefig(os.path.join(OUT, "confusion_matrix_val.png"), dpi=150)
plt.close()

# metadata
metadata = {
    "timestamp": datetime.now().isoformat(),
    "root": ROOT,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "focal_alpha": FOCAL_ALPHA,
    "focal_gamma": FOCAL_GAMMA,
    "use_amp": USE_AMP,
    "nonjunk_to_junk": NONJUNK_TO_JUNK,
    "val_frac": VAL_FRAC,
    "junk_kmax": JUNK_KMAX,
    "junk_kmeans_cap": JUNK_KMEANS_CAP,
    "target_hw": TARGET_HW,
    "seed": SEED,
    "device": device,
    "python_version": sys.version,
    "torch_version": subprocess.getoutput("python -c 'import torch; print(torch.__version__)'"),
    "gpu": subprocess.getoutput("nvidia-smi --query-gpu=name --format=csv,noheader")
}
with open(os.path.join(OUT, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

# README.md auto
readme_lines = [
f"# Junk-vs-Rest CNN Experiment",
f"**Run Time:** {metadata['timestamp']}",
"",
"## 1. Parameters",
f"- Root: `{ROOT}`",
f"- Epochs: {EPOCHS}",
f"- Batch Size: {BATCH_SIZE}",
f"- LR: {LR}",
f"- Focal (alpha, gamma): ({FOCAL_ALPHA}, {FOCAL_GAMMA})",
f"- Non-Junk : Junk Ratio = {NONJUNK_TO_JUNK}",
f"- Junk K-Means Max K = {JUNK_KMAX}",
f"- Target HW = {TARGET_HW}",
f"- Seed = {SEED}",
"",
"## 2. Hardware & Environment",
f"- Python: {metadata['python_version'].split()[0]}",
f"- Torch: {metadata['torch_version']}",
f"- GPU: {metadata['gpu']}",
"",
"## 3. Validation Metrics (Best)",
f"- Accuracy: {best['acc']:.3f}",
f"- AUC:      {best['auc']:.3f}",
f"- F1:       {best['f1']:.3f}",
"",
"## 4. Grad-CAM Center vs Border (Val sample)",
]
# append gradcam report if present
try:
    with open(os.path.join(OUT, "gradcam_center_border_summary.json")) as f:
        rep = json.load(f)
    readme_lines += [
        f"- Mean Center/Border Ratio: {rep.get('center_border_ratio_mean', None)}",
        f"- Median Center/Border Ratio: {rep.get('center_border_ratio_median', None)}",
        f"- Per-Class NotJunk: {rep.get('per_class',{}).get('notjunk', None)}",
        f"- Per-Class Junk:    {rep.get('per_class',{}).get('junk', None)}",
        f"- Samples: {rep.get('samples', 0)}",
        f"- DAPI channel index used: {rep.get('dapi_channel_index', 0)}",
    ]
except Exception as e:
    readme_lines += ["- Grad-CAM summary missing."]
readme_lines += [
"",
"## 5. Key Plots",
]
for p in sorted(glob.glob(os.path.join(OUT,"*.png"))):
    rel = os.path.basename(p)
    readme_lines.append(f"![{rel}]({rel})")
readme_lines += [
"",
"## 6. Grad-CAM PDFs",
]
for p in sorted(glob.glob(os.path.join(OUT,"*.pdf"))):
    rel = os.path.basename(p)
    readme_lines.append(f"- [{rel}]({rel})")
readme_lines += [
"",
"## 7. Notes",
"- [ ] Elbow K = ?",
"- [ ] Validation trends stable by epoch ~?",
"- [ ] Center/Border focus sufficiently > 1.5?",
"- [ ] Clusters 3–4 non-junk enriched?",
"",
"---",
"*(Generated automatically by this notebook)*",
]
with open(os.path.join(OUT,"README.md"), "w") as f:
    f.write("\n".join(readme_lines))

print("Saved: metrics_over_epochs.csv, train_curve.png, confusion_matrix_val.png, metadata.json, README.md")


Saved: metrics_over_epochs.csv, train_curve.png, confusion_matrix_val.png, metadata.json, README.md



### (Optional) Launch TensorBoard

If running locally or in an environment that supports it, you can launch TensorBoard pointing to the experiment's `tensorboard/` folder.


In [ ]:

# %load_ext tensorboard
# %tensorboard --logdir $OUT/tensorboard --port 6007
print("To view TensorBoard: tensorboard --logdir", os.path.join(OUT, "tensorboard"))


To view TensorBoard: tensorboard --logdir experiment_outputs_20251024_150445/tensorboard
